In [ ]:
# Install required packages
!pip install --upgrade google-api-python-client google-auth-httplib2 google-auth-oauthlib
!pip install pandas dateparser transformers

import pandas as pd
from datetime import datetime
from google.colab import drive
from googleapiclient.discovery import build
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request
import os.path
import pickle
import dateparser
from transformers import pipeline

SCOPES = ['https://www.googleapis.com/auth/gmail.readonly']

def gmail_authenticate():
    from google_auth_oauthlib.flow import Flow
    import json
    from IPython.display import display
    from google.colab import output
    import time

    creds = None
    token_path = '/content/drive/MyDrive/Colab Notebooks/token.pickle'

    if os.path.exists(token_path):
        with open(token_path, 'rb') as token:
            creds = pickle.load(token)
        if creds and creds.valid:
            return build('gmail', 'v1', credentials=creds)

    credentials_path = '/content/drive/MyDrive/Colab Notebooks/credentials.json'

    with open(credentials_path, 'r') as f:
        client_config = json.load(f)

    flow = Flow.from_client_config(
        client_config,
        scopes=SCOPES,
        redirect_uri='urn:ietf:wg:oauth:2.0:oob'
    )

    auth_url, _ = flow.authorization_url(prompt='consent')

    print("Please visit this URL to authorize this application:")
    print(auth_url)
    print("\nAfter authorizing, you'll get an authorization code. Paste it below.")

    auth_code = input("Enter the authorization code: ")

    flow.fetch_token(code=auth_code)
    creds = flow.credentials

    with open(token_path, 'wb') as token:
        pickle.dump(creds, token)

    return build('gmail', 'v1', credentials=creds)

def get_emails(service, max_results=10):
    results = service.users().messages().list(
        userId='me',
        maxResults=max_results
    ).execute()

    emails = []
    if 'messages' in results:
        for msg in results['messages']:
            try:
                message = service.users().messages().get(
                    userId='me',
                    id=msg['id'],
                    format='full'
                ).execute()

                headers = {h['name']: h['value'] for h in message['payload']['headers']}

                body = message.get('snippet', '')
                if 'parts' in message['payload']:
                    for part in message['payload']['parts']:
                        if part.get('mimeType') == 'text/plain' and 'data' in part.get('body', {}):
                            import base64
                            body_data = base64.urlsafe_b64decode(part['body']['data']).decode('utf-8')
                            body = body_data if body_data else body

                email_data = {
                    'id': msg['id'],
                    'sender': headers.get('From', ''),
                    'subject': headers.get('Subject', ''),
                    'date': headers.get('Date', ''),
                    'body': body
                }
                emails.append(email_data)
            except Exception as e:
                print(f"Error processing message {msg['id']}: {e}")

    return pd.DataFrame(emails)

class RelationshipManager:
    def __init__(self):
        self.contact_scores = {
            'boss@company.com': 5.0,
            'teammate@company.com': 4.0,
            'newsletter@marketing.com': 1.0
        }
        self.default_score = 2.0

    def get_score(self, sender):
        import re
        email_match = re.search(r'<(.+?)>', sender)
        email = email_match.group(1) if email_match else sender
        return self.contact_scores.get(email.lower(), self.default_score)

    def update_score(self, sender, new_interaction):
        pass

class UrgencyAnalyzer:
    def __init__(self):
        self.keywords = ['urgent', 'asap', 'immediately', 'deadline', 'important', 
                         'critical', 'emergency', 'time-sensitive', 'quick']

    def detect_urgency(self, text):
        if not text:
            return 0
        text_lower = text.lower()
        keyword_score = sum(1 for word in self.keywords if word in text_lower)
        today = datetime.today().date()
        try:
            parsed_date = dateparser.parse(text)
            if parsed_date:
                date_diff = (parsed_date.date() - today).days
                time_sensitivity = max(0, 3 - min(date_diff, 3)) if date_diff >= 0 else 0
            else:
                time_sensitivity = 0
        except Exception:
            time_sensitivity = 0
        return min(keyword_score + time_sensitivity, 5)

class ContentEvaluator:
    def __init__(self):
        try:
            self.classifier = pipeline("zero-shot-classification", 
                                     model="facebook/bart-large-mnli")
            self.labels = ["action required", "important", "urgent", "informational"]
            self.model_loaded = True
        except Exception as e:
            print(f"Could not load NLP model: {e}")
            self.model_loaded = False

    def analyze(self, text):
        if not self.model_loaded or not text:
            return 2.0
        try:
            result = self.classifier(text, self.labels)
            return max(result['scores']) * 5
        except Exception as e:
            print(f"Error in content analysis: {e}")
            return 2.0

class PriorityCalculator:
    def __init__(self):
        self.urgency_analyzer = UrgencyAnalyzer()
        self.relationship_manager = RelationshipManager()
        self.content_evaluator = ContentEvaluator()

    def calculate_priority(self, email):
        urgency = self.urgency_analyzer.detect_urgency(email['body'])
        relationship = self.relationship_manager.get_score(email['sender'])
        try:
            content = self.content_evaluator.analyze(email['body'])
        except:
            content = 2.5
        priority = (urgency * 0.5) + (relationship * 0.3) + (content * 0.2)
        return round(priority, 2)

def main():
    print("Starting Gmail NLP Analyzer...")
    try:
        service = gmail_authenticate()
        print("\u2713 Successfully authenticated with Gmail")
    except Exception as e:
        print(f"Authentication error: {e}")
        return
    try:
        print("Fetching emails...")
        emails_df = get_emails(service, max_results=20)
        print(f"\u2713 Retrieved {len(emails_df)} emails")
    except Exception as e:
        print(f"Error fetching emails: {e}")
        return
    if emails_df.empty:
        print("No emails retrieved. Please check your Gmail account and permissions.")
        return
    print("Analyzing emails...")
    prioritizer = PriorityCalculator()
    emails_df['priority'] = emails_df.apply(prioritizer.calculate_priority, axis=1)
    print("\n\u2713 Analysis complete!")
    print("\nPrioritized Emails:")
    result_df = emails_df.sort_values('priority', ascending=False)[['sender', 'subject', 'priority']]
    print(result_df.to_string())
    result_path = '/content/drive/MyDrive/Colab Notebooks/email_priorities.csv'
    result_df.to_csv(result_path)
    print(f"\nResults saved to {result_path}")

if __name__ == "__main__":
    main()
